# PiL-HQUC — Colab GPU Demo API

Notebook này **chỉ phục vụ live demo**:

```text
Frontend Vite trên máy local (:5173)
        ↓ HTTPS
Cloudflare Quick Tunnel
        ↓
FastAPI trên Google Colab (:8000)
        ↓
Qamomile → CUDA-Q → NVIDIA GPU
```

Notebook này không chạy test và không chạy benchmark, nhờ đó backend API có thể giữ GPU context ổn định trong suốt phần trình diễn.


In [ ]:
# 1) Upload and extract the project ZIP
from google.colab import files
from pathlib import Path
import os
import shutil
import zipfile

os.chdir('/content')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError(f'Upload exactly one PiL-HQUC project ZIP. Received: {zip_names}')

zip_path = Path('/content') / zip_names[0]
extract_dir = Path('/content/pil_hquc_project')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)

candidates = sorted({
    backend.parent
    for backend in extract_dir.rglob('backend')
    if backend.is_dir()
    and (backend.parent / 'frontend').is_dir()
    and (backend.parent / 'benchmark').is_dir()
})
if len(candidates) != 1:
    raise RuntimeError(f'Could not identify one project root. Candidates: {candidates}')

ROOT = candidates[0]
BACKEND = ROOT / 'backend'
FRONTEND = ROOT / 'frontend'
BENCHMARK = ROOT / 'benchmark'

print('Project ZIP:', zip_path.name)
print('Project root:', ROOT)
print('Backend:', BACKEND)
print('Frontend:', FRONTEND)
print('Benchmark:', BENCHMARK)


In [ ]:
# 2) Install the pinned CUDA-Q/Qamomile backend environment
import subprocess
import sys

requirements = BACKEND / 'requirements-quantum-colab.txt'
if not requirements.exists():
    raise FileNotFoundError(requirements)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(requirements)],
    cwd='/content',
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
print('Backend quantum dependencies installed.')


In [ ]:
# 3) Require CUDA-Q to use the Colab NVIDIA GPU
import os
import subprocess

subprocess.run(['nvidia-smi'], check=True)
os.environ['CUDAQ_TARGET'] = 'nvidia'
os.environ['REQUIRE_CUDAQ'] = '1'

import cudaq
cudaq.set_target('nvidia')
target = cudaq.get_target()
name_value = getattr(target, 'name', str(target))
target_name = name_value() if callable(name_value) else str(name_value)
print('CUDA-Q target:', target_name)
assert 'nvidia' in target_name.lower(), target_name


In [ ]:
# 4) Start the FastAPI backend on Colab port 8000
import requests
import subprocess
import sys
import time
from pathlib import Path

# Stop processes left by an earlier execution of this notebook.
for process_name in ('tunnel_process', 'api_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=8)
        except subprocess.TimeoutExpired:
            process.kill()

api_log_path = Path('/content/pil_hquc_api.log')
api_log_handle = api_log_path.open('w')

api_env = os.environ.copy()
api_env['PYTHONPATH'] = str(BACKEND)
api_env['CUDAQ_TARGET'] = 'nvidia'
api_env['REQUIRE_CUDAQ'] = '1'

api_process = subprocess.Popen(
    [
        sys.executable, '-m', 'uvicorn', 'app.main:app',
        '--host', '0.0.0.0', '--port', '8000',
    ],
    cwd=str(BACKEND),
    env=api_env,
    stdout=api_log_handle,
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    if api_process.poll() is not None:
        raise RuntimeError(api_log_path.read_text(errors='ignore'))
    try:
        response = requests.get('http://127.0.0.1:8000/api/health', timeout=2)
        if response.ok:
            print('Local API health:', response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise TimeoutError(api_log_path.read_text(errors='ignore'))


In [ ]:
# 5) Create a public HTTPS tunnel (no Cloudflare account or token required)
import platform
import re
import stat
import urllib.request

machine = platform.machine().lower()
if machine not in {'x86_64', 'amd64'}:
    raise RuntimeError(f'Unsupported Colab architecture: {machine}')

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared,
    )
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

cloudflare_log_path = Path('/content/cloudflared.log')
cloudflare_log_handle = cloudflare_log_path.open('w')
tunnel_process = subprocess.Popen(
    [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    cwd='/content',
    stdout=cloudflare_log_handle,
    stderr=subprocess.STDOUT,
)

public_url = None
url_pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
for _ in range(60):
    if tunnel_process.poll() is not None:
        raise RuntimeError(cloudflare_log_path.read_text(errors='ignore'))
    match = url_pattern.search(cloudflare_log_path.read_text(errors='ignore'))
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if not public_url:
    raise TimeoutError(cloudflare_log_path.read_text(errors='ignore'))

API_BASE_URL = public_url + '/api'
print('\nPUBLIC API BASE URL')
print(API_BASE_URL)
print('\nCreate frontend/.env.local with exactly:')
print(f'VITE_API_BASE_URL={API_BASE_URL}')


In [ ]:
# 6) Verify the public API before opening the frontend
health_url = API_BASE_URL + '/health'
response = requests.get(health_url, timeout=30)
response.raise_for_status()
print('Public API status:', response.status_code)
print(response.json())


## Chạy frontend trên máy local

Tạo `frontend/.env.local`:

```env
VITE_API_BASE_URL=https://<URL-vừa-in>.trycloudflare.com/api
```

Sau đó chạy lại Vite:

```bash
cd frontend
npm install
npm run dev
```

Mở `http://localhost:5173`.

- Không thêm dấu `/` sau `/api`.
- URL tunnel đổi sau mỗi lần Colab runtime hoặc tunnel khởi động lại.
- Giữ notebook và Colab runtime hoạt động trong suốt phần demo.
- Notebook benchmark là luồng riêng; không chạy benchmark trong runtime đang dùng để demo.


In [ ]:
# 7) Optional status check and recent API log
print('API running:', api_process.poll() is None)
print('Tunnel running:', tunnel_process.poll() is None)
print('API base URL:', API_BASE_URL)
print('\nLast API log lines:')
log_lines = api_log_path.read_text(errors='ignore').splitlines()
print('\n'.join(log_lines[-30:]))
